In [2]:
# =========================================================
# 02_dashboard_dataset.ipynb
# Create final Tableau dashboard dataset
# Output: data/cleaned/dashboard_final.csv
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# 1. Project paths
# -----------------------------

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2. Load raw files
# -----------------------------

sales = pd.read_csv(RAW_DIR / "sales.csv")
costs = pd.read_csv(RAW_DIR / "costs.csv")
waste = pd.read_csv(RAW_DIR / "waste.csv")
locations = pd.read_csv(RAW_DIR / "locations.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
vendor_prices = pd.read_csv(RAW_DIR / "vendor_prices.csv")

print("sales:", sales.shape)
print("costs:", costs.shape)
print("waste:", waste.shape)
print("locations:", locations.shape)
print("products:", products.shape)
print("vendor_prices:", vendor_prices.shape)

# -----------------------------
# 3. Aggregate sales to monthly grain
# -----------------------------

monthly_sales = (
    sales
    .groupby(["month", "location_id", "product_id"], as_index=False)
    .agg({
        "units_sold": "sum",
        "revenue": "sum",
        "discount_amount": "sum",
        "transactions_count": "sum",
        "avg_ticket": "mean"
    })
)

# -----------------------------
# 4. Merge sales + costs + waste
# -----------------------------

df = monthly_sales.merge(
    costs,
    on=["month", "location_id", "product_id"],
    how="left"
)

df = df.merge(
    waste,
    on=["month", "location_id", "product_id"],
    how="left"
)

# -----------------------------
# 5. Merge product, location, vendor data
# -----------------------------

df = df.merge(
    locations,
    on="location_id",
    how="left"
)

df = df.merge(
    products,
    on="product_id",
    how="left"
)

df = df.merge(
    vendor_prices[[
        "month",
        "product_id",
        "vendor_id",
        "vendor_name",
        "actual_unit_cost",
        "prior_unit_cost",
        "cost_change_pct"
    ]],
    on=["month", "product_id", "vendor_id"],
    how="left"
)

# -----------------------------
# 6. Clean numeric fields
# -----------------------------

numeric_cols = [
    "units_sold",
    "revenue",
    "discount_amount",
    "transactions_count",
    "avg_ticket",
    "cogs",
    "labor_cost",
    "packaging_cost",
    "overhead_allocated",
    "total_cost",
    "waste_units",
    "waste_cost",
    "menu_price",
    "standard_food_cost",
    "actual_unit_cost",
    "prior_unit_cost",
    "cost_change_pct"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# -----------------------------
# 7. Recalculate clean metrics
# -----------------------------

df["total_cost_clean"] = (
    df["cogs"]
    + df["labor_cost"]
    + df["packaging_cost"]
    + df["overhead_allocated"]
)

df["gross_profit"] = df["revenue"] - df["total_cost_clean"]

df["gross_margin_pct"] = np.where(
    df["revenue"] > 0,
    df["gross_profit"] / df["revenue"],
    0
)

df["food_cost_pct"] = np.where(
    df["revenue"] > 0,
    df["cogs"] / df["revenue"],
    0
)

df["waste_pct_revenue"] = np.where(
    df["revenue"] > 0,
    df["waste_cost"] / df["revenue"],
    0
)

df["vendor_cost_change_pct"] = df["cost_change_pct"]

df["standard_vs_actual_cost_variance"] = (
    df["actual_unit_cost"] - df["standard_food_cost"]
)

# -----------------------------
# 8. Add dashboard labels
# -----------------------------

def performance_flag(row):
    if row["gross_margin_pct"] < 0.20:
        return "Low Margin"
    elif row["gross_margin_pct"] < 0.30:
        return "Watchlist"
    else:
        return "Healthy"

df["performance_flag"] = df.apply(performance_flag, axis=1)


def priority_label(row):
    if row["gross_margin_pct"] < 0.20 and row["waste_pct_revenue"] > 0.05:
        return "High Priority"
    elif row["gross_margin_pct"] < 0.30:
        return "Medium Priority"
    else:
        return "Monitor"

df["priority_label"] = df.apply(priority_label, axis=1)

# -----------------------------
# 9. Final Tableau columns
# -----------------------------

dashboard_cols = [
    "month",
    "location_id",
    "location_name",
    "city",
    "region",
    "store_format",
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "vendor_name",
    "units_sold",
    "transactions_count",
    "revenue",
    "discount_amount",
    "cogs",
    "labor_cost",
    "packaging_cost",
    "overhead_allocated",
    "total_cost",
    "total_cost_clean",
    "gross_profit",
    "gross_margin_pct",
    "food_cost_pct",
    "waste_units",
    "waste_cost",
    "waste_pct_revenue",
    "actual_unit_cost",
    "prior_unit_cost",
    "vendor_cost_change_pct",
    "standard_vs_actual_cost_variance",
    "performance_flag",
    "priority_label"
]

missing_cols = [col for col in dashboard_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

dashboard_df = df[dashboard_cols].copy()

# -----------------------------
# 10. Quality checks
# -----------------------------

print("\nDashboard shape:", dashboard_df.shape)
print("\nMissing values:")
print(dashboard_df.isna().sum())

print("\nGross margin summary:")
print(dashboard_df["gross_margin_pct"].describe())

print("\nPerformance flags:")
print(dashboard_df["performance_flag"].value_counts())

print("\nPriority labels:")
print(dashboard_df["priority_label"].value_counts())

# -----------------------------
# 11. Export files
# -----------------------------

dashboard_output = CLEANED_DIR / "dashboard_final.csv"
summary_output = OUTPUTS_DIR / "dashboard_dataset_summary.csv"

dashboard_df.to_csv(dashboard_output, index=False)

summary_df = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "total_revenue",
        "total_cost_clean",
        "gross_profit",
        "gross_margin_pct",
        "total_waste_cost"
    ],
    "value": [
        dashboard_df.shape[0],
        dashboard_df.shape[1],
        dashboard_df["revenue"].sum(),
        dashboard_df["total_cost_clean"].sum(),
        dashboard_df["gross_profit"].sum(),
        dashboard_df["gross_profit"].sum() / dashboard_df["revenue"].sum(),
        dashboard_df["waste_cost"].sum()
    ]
})

summary_df.to_csv(summary_output, index=False)

print("\nSaved:")
print(dashboard_output)
print(summary_output)

dashboard_df.head()

sales: (190624, 9)
costs: (6272, 8)
waste: (6272, 7)
locations: (8, 6)
products: (28, 8)
vendor_prices: (784, 7)

Dashboard shape: (6272, 33)

Missing values:
month                               0
location_id                         0
location_name                       0
city                                0
region                              0
store_format                        0
product_id                          0
product_name                        0
category                            0
subcategory                         0
vendor_name                         0
units_sold                          0
transactions_count                  0
revenue                             0
discount_amount                     0
cogs                                0
labor_cost                          0
packaging_cost                      0
overhead_allocated                  0
total_cost                          0
total_cost_clean                    0
gross_profit                        0
gross

,month,location_id,location_name,city,region,store_format,product_id,product_name,category,subcategory,...,food_cost_pct,waste_units,waste_cost,waste_pct_revenue,actual_unit_cost,prior_unit_cost,vendor_cost_change_pct,standard_vs_actual_cost_variance,performance_flag,priority_label
0,2024-01,LOC001,Toronto Downtown,Toronto,Ontario,Downtown,P001,Grilled Chicken Breast,Proteins,Chicken,...,0.353928,21,100.8,0.010064,4.8,4.8,0.0,0.0,Healthy,Monitor
1,2024-01,LOC001,Toronto Downtown,Toronto,Ontario,Downtown,P002,Beef Striploin,Proteins,Beef,...,0.400669,16,110.4,0.010513,6.9,6.9,0.0,0.0,Watchlist,Medium Priority
2,2024-01,LOC001,Toronto Downtown,Toronto,Ontario,Downtown,P003,Salmon Fillet,Proteins,Fish,...,0.402434,15,106.5,0.011249,7.1,7.1,0.0,0.0,Watchlist,Medium Priority
3,2024-01,LOC001,Toronto Downtown,Toronto,Ontario,Downtown,P004,Turkey Slices,Proteins,Turkey,...,0.333132,20,84.0,0.010710,4.2,4.2,0.0,0.0,Healthy,Monitor
4,2024-01,LOC001,Toronto Downtown,Toronto,Ontario,Downtown,P005,Tofu Cubes,Proteins,Plant,...,0.286267,15,46.5,0.008614,3.1,3.1,0.0,0.0,Healthy,Monitor
